In [1]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import natural_units as nu
from scipy.special import erf
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator
from scipy.interpolate import interp1d
import re
# multi-core/thread:
import concurrent.futures

In [2]:
# HgCdTe form factors' parameters
N_q   = 800
N_E   = 300
dE    = 0.05 * nu.eV
dq    = 0.01 * nu.aEM * nu.mElectron
E_max = N_E * dE
q_max = N_q * dq
energy_gap = 0.234 * nu.eV
epsilon    = 3 * energy_gap
M_cell     = 301.74 * nu.AMU
Q_max      = np.floor((E_max - energy_gap + epsilon) / epsilon)
# Read form factor files
ff_hgte_grid = np.zeros((N_q,N_E))
ff_cdte_grid = np.zeros((N_q,N_E))
hgte_data_file = np.loadtxt('../data/form_factors/C.HgTe137.dat')
cdte_data_file = np.loadtxt('../data/form_factors/C.CdTe137.dat')
lin2_data_file = np.loadtxt('../data/form_factors/Lin2_HgCdTe.txt')
i=0
for Ei in range(N_E):
    for qi in range(N_q):
        ff_hgte_grid[qi, Ei] = hgte_data_file[i]
        ff_cdte_grid[qi, Ei] = cdte_data_file[i]
        i += 1

# correct by Lindhard:
ff_hgte_grid = ff_hgte_grid / lin2_data_file
ff_cdte_grid = ff_cdte_grid / lin2_data_file

# Interplot form factors
q_grid = np.linspace(dq, q_max, N_q)
E_grid = np.linspace(dE, E_max, N_E)

ff_hgte = RegularGridInterpolator((q_grid, E_grid), ff_hgte_grid)
ff_cdte = RegularGridInterpolator((q_grid, E_grid), ff_cdte_grid)
# To determine light or heavy mediator, just in case
mA = 0.0
def F_DM(q):
    return ((nu.aEM*nu.mElectron)**2 + mA**2)/(q**2 + mA**2)
def v_min(q, Ee, mDM):
    return Ee/q + q/2/mDM

def charge_yield(Ee, Q):
        if Ee < energy_gap:
            return 0
        else:
            Ee_1 = epsilon * (Q - 1) + energy_gap
            Ee_2 = epsilon * Q + energy_gap
            if Ee < Ee_1 or Ee > Ee_2:
                return 0.0
            else:
                return 1.0


In [3]:
# Halo DM parameters, just in case
rho_DM  = 0.3 * nu.GeV / nu.cm**3
frac_DM = 4e-3
# JWST parameters
pixel_mass = 1.2e-8 *nu.gram
exposure_time   = 3574.278 *nu.sec * 244 / 245  # since we are using (last frame - first frame)
exposure = pixel_mass * exposure_time

eta_func_dir = '../data/DM_eta_function_extended_grid'
eta_func_path = Path(eta_func_dir)
# 使用 iterdir() 遍历目录中的文件
eta_func_files = [file for file in eta_func_path.iterdir()]

In [4]:
def process_file(filename):
    file_stem = filename.stem
    if os.path.exists('../data/binned_signal_w_shield_w_lindhard_extended_grid/binned_signals_' + file_stem[9:-11] + '.txt'):
        print('File has already been processed.')
        return
    # 正则表达式匹配 mDM 和 sigma 的值
    pattern = r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2"
    # 使用 re.search() 匹配并提取
    match = re.search(pattern, file_stem)
    mDM_in_GeV = float(match.group(1))  # 第一个括号组匹配 mDM 的值
    sigma_e_in_cm2 = float(match.group(2))  # 第二个括号组匹配 sigma 的值
    mDM = mDM_in_GeV * nu.GeV
    sigma_e = sigma_e_in_cm2 * nu.cm * nu.cm
    eta_func = np.loadtxt(filename)
    eta_func_interp = interp1d(eta_func[:,0], eta_func[:,1])
    def EtaFunction(vMin):
        if vMin < eta_func[0,0] or vMin > eta_func[-1,0]:
            return 0
        else:
            return eta_func_interp(vMin).item()
    # Energy spectrum per mass:
    def dRdEe_halo(Ee, sigma_e, mDM, target):
        if target!="hgte" and target!="cdte":
            raise ValueError("target not recognized for JWST")
        elif target == "hgte":
            integral  = 0.0
            prefactor = rho_DM * frac_DM / mDM / M_cell * nu.aEM * sigma_e * nu.mElectron**2 / nu.Reduced_Mass(nu.mElectron, mDM)**2
            for qi in q_grid:
                vMin = v_min(qi, Ee, mDM)
                ff_qiE = ff_hgte((qi, Ee))
                integral += prefactor * dq / qi**2 * EtaFunction(vMin) * F_DM(qi)**2 * ff_qiE
        elif target == "cdte":
            integral  = 0.0
            prefactor = rho_DM * frac_DM / mDM / M_cell * nu.aEM * sigma_e * nu.mElectron**2 / nu.Reduced_Mass(nu.mElectron, mDM)**2
            for qi in q_grid:
                vMin = v_min(qi, Ee, mDM)
                ff_qiE = ff_cdte((qi, Ee))
                integral += prefactor * dq / qi**2 * EtaFunction(vMin) * F_DM(qi)**2 * ff_qiE
        return integral
    # Electron spectrum per mass:
    def R_Q_halo(Q, sigma_e, mDM, target):
        R_Q = 0
        for Ei in E_grid:
            cy = charge_yield(Ei, Q)
            R_Q += dE * cy * dRdEe_halo(Ei, sigma_e, mDM, target)
        return R_Q
    nq = np.zeros(30)
    total_charge = 0
    for q in range(30):
        nq_hgte = exposure * R_Q_halo(q+1, sigma_e, mDM, "hgte")
        nq_cdte = exposure * R_Q_halo(q+1, sigma_e, mDM, "cdte")
        nq[q] = np.min((nq_hgte, nq_cdte))
        total_charge += (q+1) * nq[q]
    result = [mDM_in_GeV, sigma_e_in_cm2, total_charge]
    np.savetxt('./result_extended_grid/' + file_stem[9:-11]+ '.txt', result)
    np.savetxt('../data/binned_signal_w_shield_w_lindhard_extended_grid/binned_signals_' + file_stem[9:-11] + '.txt', nq)
    print(file_stem + '   completed')

    return 0

In [5]:
# 使用 ProcessPoolExecutor 并行处理文件，限制最大进程数为 20
with concurrent.futures.ProcessPoolExecutor(max_workers=28) as executor:
    # 获取文件夹中所有的文件路径
    file_paths = [file for file in eta_func_path.iterdir() if file.is_file()]
    
    # 提交文件处理任务到进程池
    futures = {executor.submit(process_file, file): file for file in file_paths}
    
    # 逐个处理完成的任务
    for future in concurrent.futures.as_completed(futures):
        file = futures[future]
        try:
            result = future.result()  # 获取任务的返回值
            print(f"Finished processing {result}")
        except Exception as exc:
            print(f"Error processing {file}: {exc}")

eta_func_mDM=0.00177828_GeV_sigma=8.60554e-27_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.00177828_GeV_sigma=1.5303e-26_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.001_GeV_sigma=2.15505e-28_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.001_GeV_sigma=2.8738e-29_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.001_GeV_sigma=6.81485e-27_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.00177828_GeV_sigma=2.72131e-29_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.1_GeV_sigma=3.79412e-27_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.00562341_GeV_sigma=5.11243e-29_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.0562341_GeV_sigma=3.34263e-26_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.0316228_GeV_sigma=1.64988e-28_cm2_vMin=0_kps   completed
Finished processing 0
eta_func_mDM=0.316228_GeV_sigma=1.92679e-27_cm2_vMin=0_kps   completed
Finis